In [ ]:
import sys
import glob
import torch
import json
import glob
import sys
import os
from sklearn.metrics import roc_curve, auc
from scipy.stats import multivariate_normal

import seaborn as sns

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from scipy.stats import norm
from collections import defaultdict
from IPython.display import clear_output

sys.path.append('/om2/user/jmhicks/projects/TextureStreaming/code/')

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats

from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path

sys.path.append('../utls/')
sys.path.append('../src/model/')

sys.path.append("/om2/user/bjmedina/auditory-memory/memory/")

from utls.plotting import plot_dprime_by_isi, plot_itemwise_split_half_scatter_df, ensure_dir
from utls.dprime import recompute_dprime_by_isi_per_subject
from utls.reliability import compute_itemwise_split_half_reliability

from utls.loading import load_results, load_results_with_isi0_exclusion, load_results_with_isi0_dprime_exclusion, move_sequences_to_used, load_results_with_exclusion
from utls.loading import load_results_with_exclusion_2
from utls.dprime import recompute_dprime_by_isi_per_subject
from utls.reliability import compute_itemwise_split_half_reliability
from utls.plotting import plot_dprime_by_isi, plot_itemwise_split_half_scatter_df, ensure_dir

from utls.reliability import compute_power_curve
from utls.plotting import plot_power_curve

import DistanceMemoryModel
import encoders

def get_dprime_by_isi(df_per_subject, return_sem=False, return_subjects=False):
    """
    Compute mean d-prime per ISI across subjects, excluding ISI = -1 (lures).

    Args:
        df_per_subject (pd.DataFrame): Output from recompute_dprime_by_isi_per_subject.
        return_sem (bool): Whether to return standard error of the mean.
        return_subjects (bool): Whether to return per-subject d-primes too.

    Returns:
        pd.DataFrame or dict:
            If return_sem=False:
                DataFrame with columns ['isi', 'd_prime']
            If return_sem=True:
                DataFrame with columns ['isi', 'd_prime', 'sem']
            If return_subjects=True:
                Returns a dict with:
                    'summary': summary DataFrame as above,
                    'per_subject': filtered per-subject df
    """
    df_filtered = df_per_subject[df_per_subject["isi"] > -1]

    grouped = df_filtered.groupby("isi")["d_prime"]
    result_df = grouped.mean().reset_index(name="d_prime")

    if return_sem:
        result_df["sem"] = grouped.sem().values

    if return_subjects:
        return {
            "summary": result_df,
            "per_subject": df_filtered.copy()
        }

    return result_df.d_prime.tolist()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE = device
print(device)
# grabbing example list of sound
sounds_list = glob.glob("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p*/*wav")
texture_list = sounds_list

ALL_SOUNDS = glob.glob("/om2/data/public/audioset/wavs/unbalanced_train_segments_downloads/unbalanced_train_segments_downloads_*/*wav")
print(len(ALL_SOUNDS))

tasks = ["ind-nature-len120" ,"global-music-len120", "atexts-len120", "nhs-region-len120"]
which_task = tasks[2] # "global-music-len120", "atexts-len120" "nhs-region-len120"

base_path = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/{}/sequences/isi_16/len120/"

seqs_paths = {"ind-nature-len120": "mem_exp_ind-nature_2025", 
              "global-music-len120": "global-music-2025-n_80",
              "atexts-len120": "mem_exp_atexts_2025",
              "nhs-region-len120": "nhs-region-n_80"}

hr_task_name = {"ind-nature-len120": "Industrial and Nature", 
              "global-music-len120": "Globalized Music",
              "atexts-len120": "Auditory Textures",
              "nhs-region-len120": " 'Natural History of Song' "}

In [ ]:
skip = False

if skip: 
    
    exps, seqs, fnames, _ = load_results_with_exclusion_2(f"/mindhive/mcdermott/www/bjmedina/experiments/bolivia_2025/results/isi_16/{which_task}",
                                                        min_dprime=2,
                                                        min_trials=120,
                                                        skip_len60=True,
                                                        verbose=False,
                                                        return_skipped=skip)
else:
    exps, seqs, fnames = load_results_with_exclusion_2(f"/mindhive/mcdermott/www/bjmedina/experiments/bolivia_2025/results/isi_16/{which_task}",
                                                    min_dprime=2,
                                                    min_trials=120,
                                                    skip_len60=True,
                                                    verbose=False,
                                                    return_skipped=skip)



move_sequences_to_used(base_path.format(seqs_paths[which_task]), seqs)

print("Number of participants used in analysis:", len(exps))


safe_name = which_task.lower().replace(" ", "_")  # e.g., "globalized_music"
save_dir = os.path.join("/om2/user/bjmedina/auditory-memory/memory/figures/human-results/isi-16-only", safe_name)

# where you want the results
CSV_PATH = f"/om2/user/bjmedina/auditory-memory/memory/assets/lineardistancemodel/{safe_name}/"

ensure_dir(save_dir)
ensure_dir(CSV_PATH)
print(save_dir)

human_results = recompute_dprime_by_isi_per_subject(exps)
human_sensitivity = get_dprime_by_isi(human_results)

clear_output(wait=False)
CSV_PATH = f"/om2/user/bjmedina/auditory-memory/memory/assets/lineardistancemodel/{safe_name}/isi16_runs.csv"

from sklearn.metrics import mean_squared_error

# Your experiment structure (list of stimulus filepaths for each run)
experiment_list = []
for seq in seqs:
    list_to_add = []
    
    with open(base_path.format(seqs_paths[which_task]) + seq, 'r') as file:
        data = json.load(file)
   
    for stim in  data['filenames_order']:
        edited_stim_name = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/" + seqs_paths[which_task] + "/" + stim
        list_to_add.append(edited_stim_name)
    
    experiment_list.append(list_to_add)

In [ ]:
pc_dims = 256

NV = 0.2                             # per-dim noise std per drift step (your class convention)
CRIT_MULT = 1.5                          # criterion = CRIT_MULT * NV * sqrt(D)
SEED = 123                               # set to None for stochastic runs
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TIMES_TO_PLOT = [10, 17, 16, 22, 40, 80, 119]   # steps for hist panels

# Map filenames to row indices in X0
# 1) Collect all unique filenames in stable order
seen, all_files = set(), []
for seq in experiment_list:
    for fn in seq:
        if fn not in seen:
            seen.add(fn)
            all_files.append(fn)

name_to_idx = {fn: i for i, fn in enumerate(all_files)}
if which_task == "atexts-len120":

    print("using texture statistiscs")
    pc_texture_model = encoders.AudioTextureEncoder(
        statistics_dict=statistics_set.statistics,
        #pc_dims=pc_dims,
        model_params=model_params,
        sr=20000,
        rms_level=0.05,
        duration=2.0,
        device=device
    )
    
    zscore_projector = encoders.ZScoreSpace(pc_texture_model, device=device)
    zscore_projector.fit(sounds_list)

else:
    print("using nn embeddings")

    ARCHITECTURES =  ['kell2018', 'resnet50']
    TASKS = ['word', 'speaker', 'audioset', 'word_speaker_audioset']
    LAYERS = {
        'kell2018': [
            'input_after_preproc', 'relu0', 'relu1', 'relu2', 'relu3', 'relu4', 'relufc'
        ],
        'resnet50': [
            'input_after_preproc', 'conv1_relu1', 'layer1', 'layer2', 'layer3', 'layer4', 'avgpool'
        ],
    }
    
    networks = []
    for architecture in ARCHITECTURES:
        for task in TASKS:
            for layer in LAYERS[architecture]:
                networks.append({'name': f'{architecture}_{task}',
                                'layer': layer})
    
    
    model_name = ARCHITECTURES[0]
    layer = LAYERS[model_name][-1]
    task  = TASKS[3]
    
    sys.path.append(f'/om2/user/bjmedina/models/cochdnn/model_directories/{model_name}_{task}/')
    print(f'/om2/user/bjmedina/models/cochdnn/model_directories/{model_name}_{task}/')
    
    
    kell2018_layer = encoders.Kell2018Encoder(
        model_name=model_name,
        layer=layer,
        sr=20000,
        rms_level=0.05,
        duration=2.0,
        device='cuda'
    )

    print("running pca")

    pc_kell = encoders.PCASpace(
        encoder = kell2018_layer,
        n_components = 256, 
        device='cuda'
    )

    pc_kell.fit(sounds_list[:400])
    clear_output(wait=False)

    print("zscoring")

    zscore_projector = encoders.ZScoreSpace(pc_kell, device=device)
    zscore_projector.fit(sounds_list[400:])
    clear_output(wait=False)



# 2) Use your pipeline to encode clean reps X0: [M,D]
_tmp = DistanceMemoryModel.DistanceMemoryModel(
    encoding_model=zscore_projector,
    noise_variance=float(NV),   # irrelevant for encoding, but ctor requires it
    criterion=0.0,
    device=DEVICE
)
_tmp._fill_memory_bank(all_files)
clear_output(wait=False)

with torch.no_grad():
    X0 = torch.stack([rep.detach().clone().view(-1) for rep in _tmp.memory_bank], dim=0).to(DEVICE)
del _tmp
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(experiment_list[0][:10], "\n", experiment_list[1][:10])

In [ ]:
pc_texture_model = encoders.AudioTextureEncoderPCA(
    statistics_dict=statistics_set.statistics,
    pc_dims=pc_dims,
    model_params=model_params,
    sr=20000,
    rms_level=0.05,
    duration=2.0,
    device=device
)

zscore_projector = encoders.ZScoreSpace(pc_texture_model, device=device)
zscore_projector.fit(all_files)

# 2) Use your pipeline to encode clean reps X0: [M,D]
_tmp = DistanceMemoryModel.DistanceMemoryModel(
    encoding_model=zscore_projector,
    noise_variance=float(NV),   # irrelevant for encoding, but ctor requires it
    criterion=0.0,
    device=DEVICE
)
_tmp._fill_memory_bank(all_files)
clear_output(wait=False)

with torch.no_grad():
    X0 = torch.stack([rep.detach().clone().view(-1) for rep in _tmp.memory_bank], dim=0).to(DEVICE)
del _tmp
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# Compute per-feature statistics
means = X0.mean(dim=0)
stds = X0.std(dim=0)

# X0 = (X0 - means) / stds

# # Compute per-feature statistics
# means = X0.mean(dim=0)
# stds = X0.std(dim=0)

print("Mean of each feature (should be ~0):")
print(means)
plt.hist(means.detach().cpu().numpy(), label='means from data')

print("Std of each feature (should be ~1):")
print(stds)
plt.hist(stds.detach().cpu().numpy(), label='stds from data')
plt.legend();


means = torch.tensor(zscore_projector.mean)
stds  = torch.tensor(zscore_projector.std)

print("Mean of features before fit:", means.mean().item())
print("Std of features before fit:", stds.mean().item())

In [ ]:
# ---------------------------
# 1) define a common FPR grid
# ---------------------------
fpr_grid = np.linspace(0, 1, 101)  # 101 points between 0 and 1

# ---------------------------
# 2) interpolate each subject's ROC onto that gride
# ---------------------------
all_interp_tprs = []
for exp in exps:
    a = np.array(exp.response, dtype=np.int64)
    b = np.array(exp.repeat == 'true', dtype=np.int64)
    fpr, tpr, _ = roc_curve(b, a, drop_intermediate=True)
    tpr_i = np.interp(fpr_grid, fpr, tpr)  # interpolate
    all_interp_tprs.append(tpr_i)
    plt.plot(fpr, tpr, marker='o', linestyle='-', alpha=0.3)  # each participant

all_interp_tprs = np.vstack(all_interp_tprs)

# ---------------------------
# 3) group mean ± SEM on the grid
# ---------------------------
mean_human_tpr = all_interp_tprs.mean(axis=0)
sem_human_tpr  = all_interp_tprs.std(axis=0) / np.sqrt(all_interp_tprs.shape[0])
human_auc_val  = np.trapz(mean_human_tpr, fpr_grid)

# ---------------------------
# 4) plot group curve with error band
# ---------------------------
plt.plot(fpr_grid, mean_human_tpr, '-k', label=f"Mean Human ROC | AUC={human_auc_val:.3f}")
plt.fill_between(fpr_grid,
                 mean_human_tpr - sem_human_tpr,
                 mean_human_tpr + sem_human_tpr,
                 color='k', alpha=0.3)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

In [ ]:
def run_experiment(sigma0, metric="mahalanobis", debug=False):
    """
    Run recognition experiment using either Mahalanobis distance or log-likelihood scoring.

    Args:
        sigma0 (float): base standard deviation for Brownian noise
        metric (str): 'mahalanobis' or 'loglikelihood'
        debug (bool): print debug output

    Returns:
        Tuple of (hit_scores, fa_scores) depending on selected metric
    """
    assert metric in {"mahalanobis", "loglikelihood"}, f"Unknown metric: {metric}"

    hit_scores, fa_scores = [], []
    D = X0.shape[1]

    for seq in experiment_list:
        if len(seq) == 0:
            continue

        seq_idx = [name_to_idx[fn] for fn in seq]
        memory_bank = []
        bank_set = set()

        for t, idx_incoming in enumerate(seq_idx, start=1):
            probe = X0[idx_incoming].view(1, -1)

            scores = []
            for mem in memory_bank:
                age = t - mem["t_inserted"]
                if age <= 0:
                    continue
                std = max(sigma0 * np.sqrt(age), 1e-6)  # prevent div by 0
                var = std ** 2

                diff = probe - mem["mu"]
                sqdist = torch.sum(diff ** 2).item()

                if metric == "mahalanobis":
                    score = (sqdist ** 0.5) / std  # mahalanobis distance
                    scores.append(score)

                elif metric == "loglikelihood":
                    log_likelihood = (
                        -0.5 * D * np.log(2 * np.pi)
                        - D * np.log(std)
                        - 0.5 * (sqdist / var)
                    )
                    scores.append(log_likelihood)

            if scores:
                if metric == "mahalanobis":
                    score_val = min(scores)
                elif metric == "loglikelihood":
                    score_val = max(scores)

                if idx_incoming in bank_set:
                    hit_scores.append(score_val)
                else:
                    fa_scores.append(score_val)

            # Update memory bank
            memory_bank.append({
                "mu": X0[idx_incoming].detach().clone(),
                "t_inserted": t
            })
            bank_set.add(idx_incoming)

        if debug:
            print(memory_bank[0])
            print(memory_bank[-1])
            input()

    return hit_scores, fa_scores

from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np

def plot_roc_from_distances(hit_dists, fa_dists, ax=None, label=None, color=None, show=True):
    """
    Plot ROC curve from hit and false alarm distance distributions.
    Assumes lower distance = more likely 'old'.
    
    Args:
        hit_dists (array-like): Mahalanobis distances for hit trials
        fa_dists (array-like): Mahalanobis distances for false alarms
        ax (matplotlib Axes, optional): existing axis to plot into
        label (str, optional): label for legend
        color (str, optional): color for the ROC curve
        show (bool): whether to call plt.show()
    
    Returns:
        auc_score (float): area under the ROC curve
        fpr (np.ndarray): false positive rate values
        tpr (np.ndarray): true positive rate values
        thresholds (np.ndarray): thresholds used
    """
    y_true = np.concatenate([
        np.ones_like(hit_dists),     # signal
        np.zeros_like(fa_dists)      # noise
    ])
    scores = -np.concatenate([hit_dists, fa_dists])  # flip sign: lower distance = more signal-like

    fpr, tpr, thresholds = roc_curve(y_true, scores)
    auc_score = auc(fpr, tpr)

    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))

    label_str = label or f"AUC = {auc_score:.2f}"
    ax.plot(fpr, tpr, label=label_str, color=color or 'C0')
    ax.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1)
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve (Mahalanobis distance)")
    ax.grid(True)
    ax.legend()

    if show:
        plt.tight_layout()
        plt.show()

    return auc_score, fpr, tpr, thresholds

def plot_roc_from_nll(hit_dists, fa_dists, ax=None, label=None, color=None, show=True):
    """
    Plot ROC curve from hit and false alarm distance distributions.
    Assumes lower distance = more likely 'old'.
    
    Args:
        hit_dists (array-like): Mahalanobis distances for hit trials
        fa_dists (array-like): Mahalanobis distances for false alarms
        ax (matplotlib Axes, optional): existing axis to plot into
        label (str, optional): label for legend
        color (str, optional): color for the ROC curve
        show (bool): whether to call plt.show()
    
    Returns:
        auc_score (float): area under the ROC curve
        fpr (np.ndarray): false positive rate values
        tpr (np.ndarray): true positive rate values
        thresholds (np.ndarray): thresholds used
    """
    y_true = np.concatenate([
        np.ones_like(hit_dists),     # signal
        np.zeros_like(fa_dists)      # noise
    ])
    scores = np.concatenate([hit_dists, fa_dists])  # flip sign: lower distance = more signal-like

    fpr, tpr, thresholds = roc_curve(y_true, scores)
    auc_score = auc(fpr, tpr)

    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))

    label_str = label or f"AUC = {auc_score:.2f}"
    ax.plot(fpr, tpr, label=label_str, color=color or 'C0')
    ax.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1)
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve (Mahalanobis distance)")
    ax.grid(True)
    ax.legend()

    if show:
        plt.tight_layout()
        plt.show()

    return auc_score, fpr, tpr, thresholds


In [ ]:


D = X0.shape[1]

stds = np.geomspace(20, 35, 3)

for std in stds:
    lls = []
    for X in X0:
        rv = multivariate_normal(mean = X.cpu(), cov = (std**2)*np.identity(D), allow_singular=False)
        for Y in X0:
            # D = X0.shape[1]
            gaus = -rv.logpdf(Y.cpu())
            lls.append(gaus)
        break
    
    plt.hist(lls, alpha=0.3, label=f"std = {std}")
plt.legend()

In [ ]:
import numpy as np

def univariate_nll(x, mu, std, clip_std=1e-6):
    """
    Compute univariate NLLs (assuming independent Gaussians) over D dimensions.

    Args:
        x       : np.ndarray of shape (D,) — the probe
        mu      : np.ndarray of shape (D,) — the mean vector
        std     : float or np.ndarray of shape (D,) — std deviation(s)
        clip_std : float — minimum std to avoid instability

    Returns:
        total_nll : float — sum of univariate NLLs
    """
    x = np.array(x)  # ensure NumPy
    mu = np.array(mu)
    std = np.clip(std, clip_std, None)  # avoid too-small std
    var = std ** 2
    diff_sq = (x - mu) ** 2

    nll_per_dim = 0.5 * np.log(2 * np.pi) + np.log(std) + 0.5 * (diff_sq / var)
    total_nll = np.sum(nll_per_dim)

    return total_nll

D = X0.shape[1]

stds = np.geomspace(10, 1e3, 50)
means_match_debug = []
means_match = []

for std in stds:
    lls_lakshmi = []
    lls = []
    
    for X in X0:
        gaus = univariate_nll(X.cpu(), X.cpu(), std)
        #gaus = -rv.logpdf(X.cpu())
        lls_lakshmi.append(gaus)
        rv = multivariate_normal(mean = X.cpu(), cov = (std**2)*np.identity(D), allow_singular=False)
        gaus = -rv.logpdf(X.cpu())
        lls.append(gaus)
        
    means_match_debug.append(np.mean(lls_lakshmi))
    means_match.append(np.mean(lls))

plt.plot(stds, means_match, '--', label="multivariate scipy", alpha=0.4)
plt.plot(stds, means_match_debug, label="univariate independent", alpha=0.4)

plt.scatter(stds, means_match, alpha=0.4)
plt.scatter(stds, means_match_debug, alpha=0.4)

plt.ylabel("Mean NLL to self (smaller -> higher prob)")
plt.xlabel("STD")
plt.legend();

In [ ]:
#stds = np.geomspace(10, 1e9, 50)[20:]
means_nonmatch = []
means_nonmatch_debug = []

for std in stds:
    lls_lakshmi = []
    lls = []
    
    for X in X0:
        rv = multivariate_normal(mean=X.cpu(), 
                                 cov=(std**2)*np.identity(D), 
                                 allow_singular=False)

        for Y in X0:
            gaus = univariate_nll(Y.cpu(), X.cpu(), std)
            lls_lakshmi.append(gaus)
            
            gaus = -rv.logpdf(Y.cpu())
            lls.append(gaus)
        
    means_nonmatch_debug.append(np.mean(lls_lakshmi))
    means_nonmatch.append(np.mean(lls))

plt.plot(stds, means_nonmatch, '--', label="multivariate scipy", alpha=0.4)
plt.plot(stds, means_nonmatch_debug, label="univariate independent", alpha=0.4)

plt.scatter(stds, means_nonmatch, alpha=0.4)
plt.scatter(stds, means_nonmatch_debug, alpha=0.4)

plt.ylabel("Mean NLL to others (smaller -> higher prob)")
plt.xlabel("STD")
plt.legend();

In [ ]:
plt.plot(stds, np.array(means_match) - np.array(means_nonmatch))

In [ ]:
diff = np.array(means_match) - np.array(means_nonmatch)
closest_idx = np.argmin(np.abs(diff))

print(stds[closest_idx])

In [ ]:
# # # Run with Mahalanobis scoring
# # sigma0 = 1
# # hit_dists, fa_dists = run_experiment(sigma0=sigma0, metric="mahalanobis")

sigma0 = 1000
# Run with log-likelihood scoring
hit_lls, fa_lls = run_experiment(sigma0=sigma0, metric="loglikelihood")
plot_roc_from_nll(hit_lls, fa_lls);

In [ ]:
sigma0 = 10
hit_dists, fa_dists = run_experiment(sigma0=sigma0, metric="mahalanobis")
plot_roc_from_distances(hit_dists, fa_dists);

In [ ]:
plt.hist(hit_dists, alpha=0.5, bins=100);
plt.hist(fa_dists, alpha=0.5, bins=100);

In [ ]:
def run_experiment_noisy(sigma0, sigma_enc, epochs=5, metric="mahalanobis", debug=False):
    """
    Run recognition experiment using either Mahalanobis distance or log-likelihood scoring.

    Args:
        sigma0 (float): base standard deviation for Brownian noise
        metric (str): 'mahalanobis' or 'loglikelihood'
        debug (bool): print debug output

    Returns:
        Tuple of (hit_scores, fa_scores) depending on selected metric
    """
    assert metric in {"mahalanobis", "loglikelihood"}, f"Unknown metric: {metric}"

    hit_scores, fa_scores = [], []
    D = X0.shape[1]

    for _ in range(epochs):
        for seq in experiment_list:
            if len(seq) == 0:
                continue
    
            seq_idx = [name_to_idx[fn] for fn in seq]
            memory_bank = []
            bank_set = set()
    
            for t, idx_incoming in enumerate(seq_idx, start=1):
                probe = X0[idx_incoming].view(1, -1)
    
                scores = []
                for mem in memory_bank:
                    age = t - mem["t_inserted"]
                    if age <= 0:
                        continue
                    std = max(sigma0 * np.sqrt(age), 1e-6)  # prevent div by 0
                    var = std ** 2
    
                    diff = probe - mem["mu"]
                    sqdist = torch.sum(diff ** 2).item()
    
                    if metric == "mahalanobis":
                        score = (sqdist ** 0.5) / std  # mahalanobis distance
                        scores.append(score)
    
                    elif metric == "loglikelihood":
                        log_likelihood = (
                            -0.5 * D * np.log(2 * np.pi)
                            - D * np.log(std)
                            - 0.5 * (sqdist / var)
                        )
                        scores.append(log_likelihood)
    
                if scores:
                    if metric == "mahalanobis":
                        score_val = min(scores)
                    elif metric == "loglikelihood":
                        score_val = max(scores)
    
                    if idx_incoming in bank_set:
                        hit_scores.append(score_val)
                    else:
                        fa_scores.append(score_val)
    
                noise = torch.randn_like(X0[idx_incoming].detach().clone()) * sigma_enc #encoding noise
                # Update memory bank
                memory_bank.append({
                    "mu": X0[idx_incoming].detach().clone() + noise,
                    "t_inserted": t
                })
                bank_set.add(idx_incoming)
    
            if debug:
                print(memory_bank[0])
                print(memory_bank[-1])
                input()

    return hit_scores, fa_scores

def run_experiment_noisy_mean(sigma0, sigma_enc, epochs=5, metric="mahalanobis", debug=False):
    """
    Run recognition experiment using either Mahalanobis distance or log-likelihood scoring (mean score).

    Args:
        sigma0 (float): base standard deviation for Brownian noise
        metric (str): 'mahalanobis' or 'loglikelihood'
        debug (bool): print debug output

    Returns:
        Tuple of (hit_scores, fa_scores) depending on selected metric
    """
    assert metric in {"mahalanobis", "loglikelihood"}, f"Unknown metric: {metric}"

    hit_scores, fa_scores = [], []
    D = X0.shape[1]

    for _ in range(epochs):
        for seq in experiment_list:
            if len(seq) == 0:
                continue
    
            seq_idx = [name_to_idx[fn] for fn in seq]
            memory_bank = []
            bank_set = set()
    
            for t, idx_incoming in enumerate(seq_idx, start=1):
                probe = X0[idx_incoming].view(1, -1)
    
                scores = []
                for mem in memory_bank:
                    age = t - mem["t_inserted"]
                    if age <= 0:
                        continue
                    std = max(sigma0 * np.sqrt(age), 1e-6)  # prevent div by 0
                    var = std ** 2
    
                    diff = probe - mem["mu"]
                    sqdist = torch.sum(diff ** 2).item()
    
                    if metric == "mahalanobis":
                        score = (sqdist ** 0.5) / std  # mahalanobis distance
                        scores.append(score)
    
                    elif metric == "loglikelihood":
                        log_likelihood = (
                            -0.5 * D * np.log(2 * np.pi)
                            - D * np.log(std)
                            - 0.5 * (sqdist / var)
                        )
                        scores.append(log_likelihood)
    
                if scores:
                    if metric == "mahalanobis":
                        score_val = np.mean(scores)
                    elif metric == "loglikelihood":
                        score_val = np.mean(scores)
    
                    if idx_incoming in bank_set:
                        hit_scores.append(score_val)
                    else:
                        fa_scores.append(score_val)
    
                noise = torch.randn_like(X0[idx_incoming].detach().clone()) * sigma_enc #encoding noise
                # Update memory bank
                memory_bank.append({
                    "mu": X0[idx_incoming].detach().clone() + noise,
                    "t_inserted": t
                })
                bank_set.add(idx_incoming)
    
            if debug:
                print(memory_bank[0])
                print(memory_bank[-1])
                input()

    return hit_scores, fa_scores

In [ ]:
import numpy as np
# Parameters
sigma0 = 1.0   # or whatever fixed drift std you want
sigma_enc_list = np.geomspace(0.01, 5.0, 15)  # try from low to high encoding noise
metric = "mahalanobis"
epochs = 1

mean_hits, mean_fas = [], []

for sigma_enc in sigma_enc_list:
    print(f"Running with σ_enc = {sigma_enc:.3f}")
    hits, fas = run_experiment_noisy(sigma0, sigma_enc, epochs=epochs, metric=metric)
    mean_hits.append(np.mean(hits))
    mean_fas.append(np.mean(fas))

# Plot
plt.figure(figsize=(6, 4))
plt.plot(sigma_enc_list, mean_hits, marker='o', label="Mean Hit Score")
plt.plot(sigma_enc_list, mean_fas, marker='s', label="Mean FA Score")
plt.xscale('log')
plt.xlabel("Encoding Noise σ_enc (log scale)")
plt.ylabel(f"Mean Score ({metric})")
plt.title(f"Effect of Encoding Noise on Recognition ({metric})")
plt.legend()
plt.grid(True, which='both', ls='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Sweep over encoding noise values
sigma0 = 0.5
sigma_enc_list = np.geomspace(0.001, 5.0, 10)
epochs = 1
metric = "mahalanobis"  # or "mahalanobis"

plt.figure(figsize=(6, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(sigma_enc_list)))

for sigma_enc, color in zip(sigma_enc_list, colors):
    hits, fas = run_experiment_noisy(sigma0, sigma_enc, epochs=epochs, metric=metric)

    if len(hits) == 0 or len(fas) == 0:
        continue  # skip empty runs

    y_true = [1]*len(hits) + [0]*len(fas)
    y_scores = hits + fas

    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(tpr, fpr)

    label = f"σ_enc={sigma_enc:.2f}, AUC={roc_auc:.2f}"
    plt.plot(tpr, fpr, label=label, color=color)

# Plot formatting
plt.plot([0, 1], [0, 1], 'k--', lw=1)  # diagonal reference
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curves vs Encoding Noise (metric: {metric})")
plt.legend(loc="lower right", fontsize=8)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Sweep over encoding noise values
sigma0 = 0.5
sigma_enc_list = np.linspace(1, 3, 10)
epochs = 1
metric = "loglikelihood"  # or "mahalanobis"

plt.figure(figsize=(6, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(sigma_enc_list)))

for sigma_enc, color in zip(sigma_enc_list, colors):
    hits, fas = run_experiment_noisy(sigma0, sigma_enc, epochs=epochs, metric=metric)

    if len(hits) == 0 or len(fas) == 0:
        continue  # skip empty runs

    y_true = [1]*len(hits) + [0]*len(fas)
    y_scores = hits + fas

    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)

    label = f"σ_enc={sigma_enc:.2f}, AUC={roc_auc:.2f}"
    plt.plot(fpr, tpr, label=label, color=color)

# Plot formatting
plt.plot([0, 1], [0, 1], 'k--', lw=1)  # diagonal reference
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curves vs Encoding Noise (metric: {metric}) (noise σ = {sigma0})")
plt.legend(loc="lower right", fontsize=8)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Sweep over encoding noise values
sigma0 = 0.5
sigma_enc_list = np.linspace(1, 3, 10)[3:6]
epochs = 1
metric = "loglikelihood"  # or "mahalanobis"

plt.figure(figsize=(6, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(sigma_enc_list)))

for sigma_enc, color in zip(sigma_enc_list, colors):
    hits, fas = run_experiment_noisy_mean(sigma0, sigma_enc, epochs=epochs, metric=metric)

    if len(hits) == 0 or len(fas) == 0:
        continue  # skip empty runs

    y_true = [1]*len(hits) + [0]*len(fas)
    y_scores = hits + fas

    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)

    label = f"σ_enc={sigma_enc:.2f}, AUC={roc_auc:.2f}"
    plt.plot(fpr, tpr, label=label, color=color)

# Plot formatting
plt.plot([0, 1], [0, 1], 'k--', lw=1)  # diagonal reference
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curves vs Encoding Noise (metric: {metric}) (noise σ = {sigma0})")
plt.legend(loc="lower right", fontsize=8)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import sys
sys.path.append('/om2/user/jmhicks/projects/TextureStreaming/code/')
from texture_prior.utils import projection

stat_projector = projection.statistics_projector() # setup stat projector
stats_file = path.relative('../assets/texture_statistics.pt') # path to saved statistics
stats_s = torch.load(stats_file) # load in statistics
data_s = stat_projector.project(stats_s, normalize=True, nPCs=4096) # project

In [ ]:
print(data_s)

In [ ]:
# Sweep over encoding noise values
sigma0 = 0.1
sigma_enc_list = np.linspace(1, 3, 10)
epochs = 1
metric = "loglikelihood"  # or "mahalanobis"

plt.figure(figsize=(6, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(sigma_enc_list)))

for sigma_enc, color in zip(sigma_enc_list, colors):
    hits, fas = run_experiment_noisy(sigma0, sigma_enc, epochs=epochs, metric=metric)

    if len(hits) == 0 or len(fas) == 0:
        continue  # skip empty runs

    y_true = [1]*len(hits) + [0]*len(fas)
    y_scores = hits + fas

    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)

    label = f"σ_enc={sigma_enc:.2f}, AUC={roc_auc:.2f}"
    plt.plot(fpr, tpr, label=label, color=color)

# Plot formatting
plt.plot([0, 1], [0, 1], 'k--', lw=1)  # diagonal reference
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curves vs Encoding Noise (metric: {metric}) (noise σ = {sigma0})")
plt.legend(loc="lower right", fontsize=8)
plt.grid(True)
plt.tight_layout()
plt.show()